In [1]:
import numpy as np

In [5]:
#create a symmetric matrix and a random mixing matrix
#mixing matrix will be 10x4 matrix
n = 100
p = 10
q = 4
B_i = np.random.rand(p, q)
k_i = np.random.rand(n, n)
k_i = (k_i @ k_i.T)

In [6]:
#check k_i
print(np.allclose(k_i, k_i.T))

True


In [7]:
#begin lanczos
k = 3 #rank target
p = k + 10 #block size

#orthonormalize the block matrix
V1 = np.random.rand(n, p)
Q, R = np.linalg.qr(V1)
print(f"the orthonormal block matrix is: {Q}")

the orthonormal block matrix is: [[-0.10883936 -0.09578139 -0.00446767 ...  0.0653216   0.13260857
  -0.1541436 ]
 [-0.12177043 -0.07262078  0.04004826 ...  0.13119852  0.1978606
   0.14226889]
 [-0.0054674   0.02024656 -0.20691861 ... -0.19960026 -0.00760234
  -0.13582504]
 ...
 [-0.03444042  0.13924316  0.07396095 ... -0.14140868 -0.14637162
   0.13470885]
 [-0.00730762  0.10520701 -0.11123926 ... -0.13901035  0.03696837
   0.09705469]
 [-0.1400217   0.02588298  0.16013345 ... -0.15520294 -0.15689098
   0.08399676]]


In [11]:
V_j = Q
A_orig = k_i
B_j = np.zeros((p, p))
V_prev = np.zeros((n, p))
A_list = []
B_list = []
V_list = []
m = 4 * p #num blocks
for j in range(1, m+1):
    W = A_orig @ V_j - V_prev @ B_j.T
    A_j = V_j.T @ W
    A_list.append(A_j)
    W = W - V_j @ A_j
    V_prev = V_j
    V_j, B_j = np.linalg.qr(W)
    V_list.append(V_j)
    B_list.append(B_j)

T_m = np.zeros((m*p, m*p))
for j in range(m):
    T_m[j*p:(j+1)*p, j*p:(j+1)*p] = A_list[j]

for j in range(m-1):
    T_m[(j+1)*p:(j+2)*p, j*p:(j+1)*p] = B_list[j]
    T_m[j*p:(j+1)*p, (j+1)*p:(j+2)*p] = B_list[j].T

print(f"The final matrix is {T_m}")

The final matrix is [[ 1.86340599e+03 -6.36602894e+02  4.95990899e+02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-6.36602894e+02  2.26845324e+02 -1.70040336e+02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 4.95990899e+02 -1.70040336e+02  1.39667917e+02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  5.32415638e+01
   1.13171093e+01  7.83291658e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  1.13171093e+01
   1.17906940e+01  1.84726034e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  7.83291658e+00
   1.84726034e+00  9.32863828e+00]]


In [12]:
print(T_m.shape)

(676, 676)


In [18]:
theta, S = np.linalg.eigh(T_m) #eigen vetcors of T_m (mp, mp)
Q_m = np.hstack(V_list) #span of lanczos basis (n, mp)
V_hat = Q_m @ S #ritz vectors of (n, mp) shape

In [19]:
print(Q_m.shape)

(100, 676)


In [20]:
#get top k ritz values
idx = np.argsort(theta)[::-1][:k]
lambda_k = theta[idx]
V_k = V_hat[:, idx]
A_approx = V_k @ np.diag(lambda_k) @ V_k.T

In [21]:
print(f"the 3-rank approx matrix is: {A_approx}")

the 3-rank approx matrix is: [[146.7296347  -19.27048347  44.05943126 ...  77.92570526  69.65234264
   15.59483233]
 [-19.27048347  29.28108072   5.12941775 ... -39.33221749  48.70445202
   -6.76149097]
 [ 44.05943126   5.12941775  18.82661092 ...  12.19805312  55.78410505
   -2.45871213]
 ...
 [ 77.92570526 -39.33221749  12.19805312 ...  73.43308797 -19.30537649
   10.33579993]
 [ 69.65234264  48.70445202  55.78410505 ... -19.30537649 269.21470742
  -54.23920115]
 [ 15.59483233  -6.76149097  -2.45871213 ...  10.33579993 -54.23920115
   26.32678565]]


In [22]:
print(A_approx.shape)

(100, 100)
